In [ ]:
!pip install -U torch torchvision tqdm


In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.optim import SGD
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms import functional as F
import kagglehub


#runpod specific paths to save to persistent volume 
WORKSPACE = "/workspace"
CKPT_DIR = os.path.join(WORKSPACE, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

DATASET_PATH = kagglehub.dataset_download("rawsi18/military-assets-dataset-12-classes-yolo8-format")

print("Path to dataset files:", DATASET_PATH)


def find_directory(base_path, target_name):
    """Recursively find a directory by name"""
    for root, dirs, files in os.walk(base_path):
        if target_name in dirs:
            return os.path.join(root, target_name)
    return None

train_base = find_directory(DATASET_PATH, "train")
val_base = find_directory(DATASET_PATH, "valid") or find_directory(DATASET_PATH, "val")


print(f"\nFound train directory: {train_base}")
print(f"Found validation directory: {val_base}")

TRAIN_IMG_DIR = os.path.join(train_base, "images")
TRAIN_LABEL_DIR = os.path.join(train_base, "labels")
VAL_IMG_DIR = os.path.join(val_base, "images")
VAL_LABEL_DIR = os.path.join(val_base, "labels")

for path, name in [(TRAIN_IMG_DIR, "Train images"), (TRAIN_LABEL_DIR, "Train labels"),
                    (VAL_IMG_DIR, "Val images"), (VAL_LABEL_DIR, "Val labels")]:
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
        print(f"{name}: {count} files")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 13  #
BATCH_SIZE = 4
MAX_EPOCHS = 40
LR = 0.005

class YoloToRCNNDataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = sorted(
            f for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(
            self.label_dir, os.path.splitext(img_name)[0] + ".txt"
        )

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    cls, cx, cy, bw, bh = map(float, line.split()[:5])
                    xmin = (cx - bw / 2) * w
                    ymin = (cy - bh / 2) * h
                    xmax = (cx + bw / 2) * w
                    ymax = (cy + bh / 2) * h
                    if xmax > xmin and ymax > ymin:
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(int(cls) + 1)

        if boxes:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }

        return F.to_tensor(img), target

def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = YoloToRCNNDataset(
    img_dir=TRAIN_IMG_DIR,
    label_dir=TRAIN_LABEL_DIR,
)

val_ds = YoloToRCNNDataset(
    img_dir=VAL_IMG_DIR,
    label_dir=VAL_LABEL_DIR,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=4
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=4
)

print(f"\nDataset loaded: {len(train_ds)} train images, {len(val_ds)} val images")

# default coco pretrained faster rcnn 

model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)


# sgd optimizer used could try adam here in the future ? 
optimizer = SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, momentum=0.9, weight_decay=5e-4
)



# MAP for evals 
metric = MeanAveragePrecision(
    box_format="xyxy", iou_type="bbox", class_metrics=True
)

best_map = 0.0
best_ckpt = os.path.join(CKPT_DIR, "faster_rcnn_best.pth")



# training loop 

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS}"):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        
        
        
    # disables dropout all neurons are stable 
    model.eval()
    
    metric.reset()

    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            outputs = model(images)

            for o, t in zip(outputs, targets):
                preds = {
                    "boxes": o["boxes"].cpu(),
                    "scores": o["scores"].cpu(),
                    "labels": o["labels"].cpu() - 1,
                }
                gts = {
                    "boxes": t["boxes"].cpu(),
                    "labels": t["labels"].cpu() - 1,
                }
                metric.update([preds], [gts])

    results = metric.compute()
    val_map = results["map_50"].item()

    print(f"Epoch {epoch+1}: loss={epoch_loss:.4f}, val mAP@0.5={val_map:.4f}")

    if val_map > best_map:
        best_map = val_map
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_map50": val_map,
            },
            best_ckpt,
        )
        print("Saved new best checkpoint")

print("Training complete.")